In [1]:

from pathlib import Path
import pandas as pd

# Set the project and Bengaluru dataset folders
PROJECT_DIR = Path(r"D:\RouteRanker")
BENGALURU_DIR = PROJECT_DIR / "data" / "raw" / "bengaluru"

# Find the folder containing the GTFS files
matches = list(BENGALURU_DIR.rglob("routes.txt"))

if not matches:
    print("ERROR: routes.txt was not found. Please check the extracted folder.")
else:
    GTFS_DIR = matches[0].parent
    print("GTFS folder:", GTFS_DIR)

    # Load the main GTFS files as text
    files_to_load = [
        "routes.txt",
        "stops.txt",
        "trips.txt",
        "stop_times.txt"
    ]

    data = {}

    for filename in files_to_load:
        filepath = GTFS_DIR / filename

        if filepath.exists():
            data[filename.replace(".txt", "")] = pd.read_csv(
                filepath, dtype=str
            )
            print(f"{filename}: {data[filename.replace('.txt', '')].shape}")
        else:
            print(f"WARNING: {filename} was not found.")


GTFS folder: D:\RouteRanker\data\raw\bengaluru\bus
routes.txt: (4434, 5)
stops.txt: (9960, 9)
trips.txt: (57836, 6)
stop_times.txt: (1540686, 6)


In [2]:

# Check missing values in each GTFS file

for name, df in data.items():
    print(f"\n{'=' * 50}")
    print(f"File: {name}.txt")
    print(f"Total rows: {len(df)}")
    print(f"Total missing values: {df.isnull().sum().sum()}")

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("No missing values detected.")
    else:
        print("Columns containing missing values:")
        print(missing.sort_values(ascending=False))



File: routes.txt
Total rows: 4434
Total missing values: 0
No missing values detected.

File: stops.txt
Total rows: 9960
Total missing values: 30368
Columns containing missing values:
location_type     9948
parent_station    9883
platform_code     9883
stop_desc          455
zone_id            199
dtype: int64

File: trips.txt
Total rows: 57836
Total missing values: 0
No missing values detected.

File: stop_times.txt
Total rows: 1540686
Total missing values: 0
No missing values detected.


In [3]:

# Deeper data-quality checks for Bengaluru GTFS

# 1. Check empty strings and whitespace-only cells
print("EMPTY / WHITESPACE-ONLY CELLS")

for name, df in data.items():
    blank_counts = df.apply(
        lambda col: col.str.strip().eq("").sum()
        if col.dtype == "object" else 0
    )
    blank_counts = blank_counts[blank_counts > 0]

    print(f"\n{name}.txt")
    if blank_counts.empty:
        print("No empty-string cells detected.")
    else:
        print(blank_counts)


# 2. Check duplicate IDs
print("\n\nDUPLICATE ID CHECKS")

id_columns = {
    "routes": ["route_id"],
    "stops": ["stop_id"],
    "trips": ["trip_id"],
}

for name, columns in id_columns.items():
    df = data.get(name)
    if df is None:
        continue

    for column in columns:
        if column in df.columns:
            print(
                f"{name}.{column}: "
                f"{df[column].duplicated().sum()} duplicate values"
            )


# 3. Check duplicate stop-time records
stop_times = data.get("stop_times")

if stop_times is not None:
    print(
        "\nExact duplicate stop_times rows:",
        stop_times.duplicated().sum()
    )

    key_columns = ["trip_id", "stop_sequence"]
    if all(c in stop_times.columns for c in key_columns):
        print(
            "Duplicate trip_id + stop_sequence combinations:",
            stop_times.duplicated(subset=key_columns).sum()
        )


# 4. Check relationships between GTFS files
routes = data.get("routes")
stops = data.get("stops")
trips = data.get("trips")

if routes is not None and trips is not None:
    if {"route_id"}.issubset(routes.columns) and {
        "route_id"
    }.issubset(trips.columns):
        missing_routes = (~trips["route_id"].isin(
            routes["route_id"]
        )).sum()
        print("\nTrips referencing missing routes:", missing_routes)

if trips is not None and stop_times is not None:
    if "trip_id" in trips.columns and "trip_id" in stop_times.columns:
        missing_trips = (~stop_times["trip_id"].isin(
            trips["trip_id"]
        )).sum()
        print("Stop times referencing missing trips:", missing_trips)

        trips_without_times = (
            ~trips["trip_id"].isin(stop_times["trip_id"])
        ).sum()
        print("Trips without stop-time records:", trips_without_times)

if stops is not None and stop_times is not None:
    if "stop_id" in stops.columns and "stop_id" in stop_times.columns:
        missing_stops = (~stop_times["stop_id"].isin(
            stops["stop_id"]
        )).sum()
        print("Stop times referencing missing stops:", missing_stops)


EMPTY / WHITESPACE-ONLY CELLS

routes.txt
No empty-string cells detected.

stops.txt
No empty-string cells detected.

trips.txt
No empty-string cells detected.

stop_times.txt
No empty-string cells detected.


DUPLICATE ID CHECKS
routes.route_id: 0 duplicate values
stops.stop_id: 0 duplicate values
trips.trip_id: 0 duplicate values

Exact duplicate stop_times rows: 0
Duplicate trip_id + stop_sequence combinations: 0

Trips referencing missing routes: 0
Stop times referencing missing trips: 0
Trips without stop-time records: 0
Stop times referencing missing stops: 0


In [4]:

# Inventory all GTFS text files
for path in sorted(GTFS_DIR.glob("*.txt")):
    name = path.stem

    if name in data:
        df = data[name]
    else:
        df = pd.read_csv(path, dtype=str)

    print(f"\nFile: {path.name}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", list(df.columns))
    print("Missing values:", int(df.isna().sum().sum()))



File: agency.txt
Rows: 1
Columns: 5
Column names: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang']
Missing values: 0

File: attributions.txt
Rows: 2
Columns: 6
Column names: ['attribution_id', 'organization_name', 'is_producer', 'is_operator', 'is_authority', 'attribution_url']
Missing values: 3

File: calendar.txt
Rows: 1
Columns: 10
Column names: ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date', 'service_id']
Missing values: 0

File: fare_attributes.txt
Rows: 110
Columns: 7
Column names: ['fare_id', 'price', 'currency_type', 'payment_method', 'transfers', 'transfer_duration', 'agency_id']
Missing values: 0

File: fare_rules.txt
Rows: 170161
Columns: 4
Column names: ['fare_id', 'route_id', 'origin_id', 'destination_id']
Missing values: 45759

File: feed_info.txt
Rows: 1
Columns: 8
Column names: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version'


## Dataset Relationships

This dataset follows the General Transit Feed Specification (GTFS), which organizes public transit information into related files.

- **routes.txt:** Contains route information. Routes are identified by `route_id`.
- **trips.txt:** Contains scheduled trips. The `route_id` connects a trip to a route, while `service_id` identifies its service schedule.
- **stop_times.txt:** Contains scheduled stop times and stop order for each trip. The `trip_id` connects records to trips, and `stop_id` connects them to stops.
- **stops.txt:** Contains stop identifiers, names, and available location information.
- **calendar.txt:** When present, defines regular service days and operating periods.
- **calendar_dates.txt:** When present, specifies additions or exceptions to regular service dates.

### Relationship Validation

The notebook checks whether trips reference valid routes, stop-time records reference valid trips and stops, and trips have stop-time records. Duplicate identifiers and missing values are also examined.
